# Iteration 1 — Geometric Baseline

**Status:** experimental baseline.

This notebook records the initial lightweight 2D-to-3D reconstruction approach. It is retained as the baseline for comparison with later learned multi-view approaches.

## Pipeline

`RGB image → foreground segmentation → preprocessing → approximate depth/height field → 3D point generation → triangular mesh`

In [ ]:
# Environment setup
!pip install -q diffusers transformers accelerate gradio trimesh pyvista open3d rembg onnxruntime einops kornia
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

import torch
import onnxruntime as ort

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
print(f"ONNX Runtime Version: {ort.__version__}")

In [ ]:
# Baseline reconstruction engine
import os
import torch
import numpy as np
import PIL.Image as Image
import rembg
import trimesh
from diffusers import DiffusionPipeline
import gradio as gr

class MultiView3DEngine:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Initializing 3D Engine on {self.device}...")
        
        # Load Zero123++ diffusion model
        self.pipe = DiffusionPipeline.from_pretrained(
            "sudo-ai/zero123plus-v1.1",
            custom_pipeline="sudo-ai/zero123plus-pipeline",
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
        )
        if self.device == "cuda":
            self.pipe.to("cuda")
            
    def preprocess_image(self, image: Image.Image) -> Image.Image:
        """Removes background and extracts clean RGBA image."""
        image = image.convert("RGB")
        no_bg = rembg.remove(image)
        no_bg = no_bg.resize((320, 320))
        return no_bg

    def generate_multiview(self, input_img: Image.Image):
        """Generates cross-view synthetic angles."""
        processed_img = self.preprocess_image(input_img)
        
        with torch.autocast("cuda") if self.device == "cuda" else torch.no_grad():
            result = self.pipe(
                image=processed_img,
                num_inference_steps=28,
                guidance_scale=4.0
            ).images[0]
            
        return processed_img, result

    def generate_3d_mesh(self, input_image: Image.Image):
        """Builds a fully colored, masked 3D mesh with proper vertex lighting."""
        if input_image is None:
            return None, "Please upload an image."

        try:
            processed_img, _ = self.generate_multiview(input_image)
            
            # Extract RGBA array for colors & alpha mask
            rgba_np = np.array(processed_img)
            rgb_np = rgba_np[:, :, :3]
            alpha_mask = rgba_np[:, :, 3] > 50  # Mask out background
            
            h, w = alpha_mask.shape
            x, y = np.meshgrid(np.arange(w), np.arange(h))
            
            # Estimate 3D depth based on luminance intensity
            gray_img = np.array(processed_img.convert("L"))
            z = (255 - gray_img) / 255.0 * 0.4  
            
            # Center coordinates & scale depth
            vertices = np.column_stack([x.ravel() - w/2, -(y.ravel() - h/2), z.ravel() * 80])
            
            # Map RGB colors directly to 3D Vertices (Prevents Black & White appearance)
            vertex_colors = np.column_stack([rgb_np.reshape(-1, 3), np.full((w * h, 1), 255)])
            
            # Construct triangular faces ONLY for visible object pixels
            faces = []
            for i in range(h - 1):
                for j in range(w - 1):
                    # Check if all 4 corners belong to the object mask
                    if alpha_mask[i, j] and alpha_mask[i+1, j] and alpha_mask[i, j+1] and alpha_mask[i+1, j+1]:
                        idx = i * w + j
                        faces.append([idx, idx + 1, idx + w])
                        faces.append([idx + 1, idx + w + 1, idx + w])
                        
            if len(faces) == 0:
                return None, "Error: Could not isolate foreground object."

            # Create Trimesh object with vertex colors and surface normals
            mesh = trimesh.Trimesh(
                vertices=vertices, 
                faces=faces, 
                vertex_colors=vertex_colors,
                process=True
            )
            
            # Apply light surface smoothing to remove pixelation steps
            trimesh.smoothing.filter_laplacian(mesh, iterations=2)
            
            # Fix surface winding & normals for correct 3D light reflection
            mesh.fix_normals()
            
            # Export as GLB format (preserves colors/textures better than standard OBJ in Gradio)
            output_path = "/kaggle/working/output_3d_model.glb"
            mesh.export(output_path)
            
            return output_path, "3D Textured Model Generated Successfully!"
            
        except Exception as e:
            return None, f"Execution Error: {str(e)}"

# Re-initialize engine instance
engine = MultiView3DEngine()

In [ ]:
# Optional Gradio interface

def process_and_render(image):
    mesh_path, status = engine.generate_3d_mesh(image)
    return mesh_path, status

# Build Gradio Block UI
with gr.Blocks(title="Multi-View Image to 3D Reconstruction Engine") as demo:
    gr.Markdown(
        """
        # Multi-View Image to 3D Reconstruction Engine
        Upload a photo of an object (front or perspective view) to generate a 3D Mesh (`.obj`) directly on Kaggle GPU.
        """
    )
    
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="pil", label="Upload Object Image")
            submit_btn = gr.Button("Generate 3D Model", variant="primary")
            status_output = gr.Textbox(label="Status / Diagnostics")
            
        with gr.Column():
            output_3d = gr.Model3D(label="Interactive 3D Preview (Rotate / Zoom)", clear_color=[0.8, 0.8, 0.8, 1.0])

    submit_btn.click(
        fn=process_and_render,
        inputs=[input_img],
        outputs=[output_3d, status_output]
    )

# Launch with share=True to generate live public URL on Kaggle
demo.queue().launch(share=True, debug=True)